# 02 · Preprocessing and data splits

Builds the labels, splits the patients into train / validation / test, and
fits the tabular preprocessing on the training patients only. Everything
downstream reads the files written here.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "angiocad_common.py").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import angiocad_common as ac
import json
import pickle
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore", message="Conditional Formatting extension")
ac.ensure_dirs()
OUT = ac.OUTPUT_DIR

## Labels and cohort

In [3]:
cohort, report = ac.build_cohort()
print(json.dumps(report, indent=2))
if report["unparsed_values"]:
    raise ValueError(f"Unreadable stenosis grades, fix these first: {report['unparsed_values']}")

print(f"\nCohort: {len(cohort)} patients, CAD prevalence {cohort['has_cad'].mean():.3f}")
cohort.to_csv(OUT / "clinical_features_original.csv", index=False)

{
  "n_labelled": 413,
  "n_without_any_grade": 0,
  "n_with_some_missing_segments": 0,
  "unparsed_values": [],
  "n_with_clinical": 377,
  "n_cohort": 377
}

Cohort: 377 patients, CAD prevalence 0.790


## Model inputs
Only information available before the angiogram is read goes into the
model. The Gensini score is derived from the angiogram itself, and
`Diagnosis` / `IHD` are kept for subgroup analysis, so none of them is an input.

In [4]:
numeric = [c for c in ac.NUMERIC_FEATURES if c in cohort.columns]
categorical = [c for c in ac.CATEGORICAL_FEATURES if c in cohort.columns]
absent = sorted(set(ac.NUMERIC_FEATURES + ac.CATEGORICAL_FEATURES) - set(cohort.columns))
if absent:
    # An earlier run lost BUN this way without any message.
    raise KeyError(f"Expected feature columns are missing from the spreadsheet: {absent}")
assert not set(numeric + categorical) & set(ac.LEAKY_COLUMNS)
print(f"{len(numeric)} numeric and {len(categorical)} categorical inputs")

22 numeric and 17 categorical inputs


## Patient-level split
68 / 12 / 20, stratified on the label. One patient contributes one row, so
all frames of a patient end up in the same split.

In [5]:
ids = cohort["ID"].to_numpy()
y = cohort["has_cad"].to_numpy()

trainval_ids, test_ids = train_test_split(ids, test_size=0.20, stratify=y, random_state=ac.SEED)
y_trainval = cohort.set_index("ID").loc[trainval_ids, "has_cad"].to_numpy()
train_ids, val_ids = train_test_split(trainval_ids, test_size=0.15, stratify=y_trainval,
                                      random_state=ac.SEED)
splits = {"train": sorted(map(int, train_ids)), "val": sorted(map(int, val_ids)),
          "test": sorted(map(int, test_ids))}

assert not set(splits["train"]) & set(splits["val"])
assert not set(splits["train"]) & set(splits["test"])
assert not set(splits["val"]) & set(splits["test"])
assert sum(map(len, splits.values())) == len(cohort)

by_id = cohort.set_index("ID")
for name, s in splits.items():
    yy = by_id.loc[s, "has_cad"]
    print(f"{name:5s}: {len(s):3d} patients   CAD {int(yy.sum()):3d}   no CAD {int((yy == 0).sum()):3d}")

train: 255 patients   CAD 202   no CAD  53
val  :  46 patients   CAD  36   no CAD  10
test :  76 patients   CAD  60   no CAD  16


## Preprocessing, fitted on the training split
Outliers are clipped to the training 1st-99th percentile, missing values are
imputed (median / most frequent), numeric columns are standardised and
categorical ones one-hot encoded.

In [6]:
pre = ac.build_preprocessor(numeric, categorical)
X = {name: by_id.loc[s, numeric + categorical] for name, s in splits.items()}
pre.fit(X["train"])

feature_names = [ac.clean_feature_name(n) for n in pre.get_feature_names_out()]
assert len(set(feature_names)) == len(feature_names), "feature names must be unique"

tables = {}
for name, s in splits.items():
    t = pd.DataFrame(pre.transform(X[name]), columns=feature_names)
    t.insert(0, "ID", s)
    t["has_cad"] = by_id.loc[s, "has_cad"].to_numpy()
    assert np.isfinite(t[feature_names].to_numpy()).all()
    t.to_csv(OUT / f"clinical_features_{name}.csv", index=False)
    tables[name] = t
    print(f"{name:5s}: {t.shape}")

train: (255, 41)
val  : (46, 41)
test : (76, 41)


In [7]:
# Labels and descriptors that are not model inputs, one row per patient.
meta_cols = ["has_cad", "max_stenosis", "num_diseased_vessels", "num_diseased_segments",
             "severity", "Diagnosis", "Sex", "Age"]
split_of = {i: name for name, s in splits.items() for i in s}
metadata = cohort[["ID"] + meta_cols].copy()
metadata.insert(1, "split", metadata["ID"].map(split_of))
metadata.to_csv(OUT / "clinical_metadata.csv", index=False)

with open(OUT / "splits.json", "w") as f:
    json.dump(splits, f, indent=2)

weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=tables["train"]["has_cad"].to_numpy())
class_weights = {"0": float(weights[0]), "1": float(weights[1])}
with open(OUT / "class_weights.json", "w") as f:
    json.dump(class_weights, f, indent=2)

with open(OUT / "feature_dim.json", "w") as f:
    json.dump({"n_features_out": len(feature_names), "feature_names_out": feature_names,
               "numeric_cols_in": numeric, "categorical_cols_in": categorical,
               "seed": ac.SEED}, f, indent=2)

# The fitted preprocessor uses a class from angiocad_common, so src/ must be
# on the path when this file is unpickled.
with open(OUT / "preprocessing_artifacts.pkl", "wb") as f:
    pickle.dump({"preprocessor": pre, "numeric_cols": numeric, "categorical_cols": categorical,
                 "feature_names_out": feature_names, "seed": ac.SEED}, f)

print(f"{len(feature_names)} model inputs")
print(f"class weights (training split): {class_weights}")

39 model inputs
class weights (training split): {'0': 2.4056603773584904, '1': 0.6311881188118812}


## Checks on what was written

In [8]:
check = {n: pd.read_csv(OUT / f"clinical_features_{n}.csv") for n in splits}

for n, t in check.items():
    assert sorted(t["ID"]) == splits[n], f"{n}: IDs differ from splits.json"
    assert t.drop(columns=["ID", "has_cad"]).columns.tolist() == feature_names
    assert t.select_dtypes(exclude="number").empty, f"{n}: non-numeric column"

# The scaler saw training rows only: training means are ~0, the others are not forced to be.
num_out = feature_names[:len(numeric)]
print("largest |mean| of scaled numeric inputs:",
      {n: round(float(t[num_out].mean().abs().max()), 3) for n, t in check.items()})

clip = pre.named_transformers_["num"].named_steps["clip"]
limits = pd.DataFrame({"lower": clip.lo_, "upper": clip.hi_}, index=numeric).round(2)
limits.to_csv(ac.RESULT_DIR / "clipping_limits.csv")
print("\nClipping limits learned on the training split:")
print(limits.to_string())
print("\nAll checks passed.")

largest |mean| of scaled numeric inputs: {'train': 0.0, 'val': 0.319, 'test': 0.323}

Clipping limits learned on the training split:
                    lower     upper
Age                 33.00     85.49
Length             146.07    188.64
Weight              52.88    110.00
BMI                 19.90     38.77
LVEF                 0.15      0.60
LVEF dysfunction     1.00      4.00
TG                  43.00    513.23
Cholesterol         86.51    277.96
HDL                 25.00     71.96
LDL                 34.52    179.40
FBS                 75.47    388.85
Cr                   0.70      5.62
BUN                 15.00     99.27
Hgb                  8.00     18.55
Hct                 24.38     54.26
leukocytes        4600.00  20084.00
Neut                 0.32      0.94
Monocyte             0.00      0.09
MXD                  0.01      0.15
Lymph                0.04      0.54
PLT                 72.78    422.06
RDW                 11.66     17.79

All checks passed.
